# Glacier outputs postprocessing

In [ ]:
# basic
import os
import yaml
from pathlib import Path
from tqdm import tqdm

# spatial
import geopandas as gpd
import pandas as pd
import xarray as xr 

wd = Path('/home/rooda/OneDrive/Projects/Whatif-Andes/')
os.chdir(wd) 

# import config and local module
with open("config.yml") as f:
    cfg = yaml.safe_load(f)

# local utils
from src import glaciers

In [ ]:
# lookup tables
andeangc_lookup   = gpd.read_file(cfg["dir"]["andeangc"] + "dataset/AndeanGC_shape.gpkg").set_index('gauge_id')[["geometry"]]
hydroatlas_lookup = gpd.read_file("data/hydro/basins_hydroATLAS_lev08.gpkg").set_index('gauge_id')[["geometry"]]

glaciers_lookup = pd.concat([gpd.read_file(cfg["dir"]["data"] + "/GLACIERS/OUTLINES/RGI60_16.shp").set_index('RGIId')[["geometry"]], 
                             gpd.read_file(cfg["dir"]["data"] + "/GLACIERS/OUTLINES/RGI60_17.shp").set_index('RGIId')[["geometry"]]])

## Region-wide aggregation

In [ ]:
# period 2000-2024
df_historical = xr.open_dataset(cfg["dir"]["pipeline"] + "/oggm15/run_output_historical.nc")[["melt_on_glacier", "volume"]]
df_historical = df_historical.drop_vars(["hydro_month", "calendar_month",]).sum(dim="rgi_id").drop_vars(["hydro_year", "calendar_year"])
df_historical.to_netcdf("data/glacier/glacier_evolution_2000_2024.nc")

df_future = []
files = list(Path(cfg["dir"]["pipeline"]).glob("oggm15/*output*"))
files = [f for f in files if "historical" not in f.name]
warming_levels = [f.name.split('_')[2] for f in files]

# period 2000-2099
for gcm in files:
    ts_future = xr.open_dataset(gcm)[["melt_on_glacier", "volume"]].load()
    ts_future = ts_future.drop_vars(["hydro_month", "calendar_month", "hydro_year", "calendar_year"])
    ts_future = ts_future.sum(dim="rgi_id")
    df_future.append(ts_future)

df_future = xr.concat(df_future, dim = "warming_level")
df_future = df_future.assign_coords(warming_level=("warming_level", warming_levels))
df_future.to_netcdf("data/glacier/glacier_evolution_2025_2099.nc")

## Basin-wide aggregation

In [ ]:
hydroatlas_lookup = glaciers.calculate_basin_overlap(glaciers_lookup, hydroatlas_lookup)
andeangc_lookup   = glaciers.calculate_basin_overlap(glaciers_lookup, andeangc_lookup)

In [ ]:
paths = list(Path(cfg["dir"]["pipeline"]).glob("oggm15/*output_historical*nc"))

for path in tqdm(paths):  

    # Load and preprocess data once
    oggm_ts = xr.open_dataset(path)[glaciers.variables]
    oggm_ts = oggm_ts.sel(time = slice(2000, 2023)) # TODO 2024 with no data
    oggm_ts = glaciers.change_calendar(oggm_ts)
    oggm_ts = glaciers.hydro_variables(oggm_ts, only_melt=True)
    
    # for HydroATLAS
    hydroatlas_lookup = hydroatlas_lookup[hydroatlas_lookup.index.isin(oggm_ts.rgi_id.values)]
    oggm_ts_hydroatlas = glaciers.aggregate_data_by_basin(oggm_ts, hydroatlas_lookup)
    oggm_ts_hydroatlas = oggm_ts_hydroatlas.astype('float32')
    oggm_ts_hydroatlas.to_netcdf("data/glacier/HydroAtlas/" + path.name, encoding={var: {"compression": "zstd"} for var in oggm_ts_hydroatlas.data_vars})

    # for AndeanGC
    andeangc_lookup = andeangc_lookup[andeangc_lookup.index.isin(oggm_ts.rgi_id.values)]
    oggm_ts_andeangc = glaciers.aggregate_data_by_basin(oggm_ts, andeangc_lookup)
    oggm_ts_andeangc = oggm_ts_andeangc.astype('float32')
    oggm_ts_andeangc.to_netcdf("data/glacier/AndeanGC/" + path.name, encoding={var: {"compression": "zstd"} for var in oggm_ts_andeangc.data_vars})